# View LSSTCam DeepCoadd Cutouts in Firefly

- **Author:** Sylvie Dagoret-Campagne
- **affiliation** : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-25
- **Last update:** 2026-06-25


## Import

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.axes_grid1 import make_axes_locatable

from astropy.visualization import ImageNormalize, AsinhStretch, PercentileInterval, ZScaleInterval
from astropy.wcs import WCS
from astropy.time import Time
import astropy.units as u


# import lsst.daf.butler as dafButler
from lsst.daf.butler import Butler

import lsst.geom as geom
import lsst.sphgeom as sphgeom
from lsst.geom import Box2I, Point2I, Extent2I, Point2D


from lsst.geom import SpherePoint, degrees
import lsst.afw.display as afwDisplay
from firefly_client import FireflyClient
import firefly_client.plot as ffplt

from lsst.skymap import PatchInfo, Index2D

In [ ]:
import gc

In [ ]:
def dataset_type_exists(butler, name):
    try:
        butler.registry.getDatasetType(name)
        return True
    except KeyError:
        return False

In [ ]:
def get_first_existing_dataset(butler, dataset_names, dataId, collections=None):
    for name in dataset_names:
        try:
            obj = butler.get(name, dataId, collections=collections)
            print(f"✔ Found dataset: {name}")
            return obj, name
        except Exception:
            continue

    raise ValueError("No valid dataset found among candidates.")

## Configuration

**Edit only this cell** to select the target field, the Butler collection,
and the band to display. Everything else runs automatically.


In [ ]:
# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "DEEPCCUTOUTS_01"
DIR_DATA_IN = f"data_{NB_TAG}_in"
DIR_DATA_OUT = f"data_{NB_TAG}_out"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA_IN, exist_ok=True)
os.makedirs(DIR_DATA_OUT, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data input: {os.path.abspath(DIR_DATA_IN)}")
print(f"Data output: {os.path.abspath(DIR_DATA_IN)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure to PDF and PNG inside DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

In [ ]:
# ============================================================
# Input file which contains the targets
# ============================================================
target_file = "summary_visit_counts_per_star.csv"
df_targets = pd.read_csv(os.path.join(DIR_DATA_IN, target_file))

In [ ]:
df_targets

In [ ]:
# ============================================================
# USER CONFIGURATION — edit only this cell to change the target
# ============================================================

# --- Butler repository and collections ---
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

instrument = "LSSTCam"
skymapName = "lsst_cells_v2"

# --- Field selection ---
# Choose from the dictionaries defined in the next cell.
# Set field_dict to either lsstcam_targets or lsstcam_fields,
# then set key to the corresponding field name.
#
# lsstcam_targets keys : "47 Tuc", "Rubin SV 38 7", "Fornax dSph",
#                        "ECDFS", "EDFS", "Rubin SV 95 -25", "Seagull"
# lsstcam_fields  keys : "COSMOS", "XMM-LSS", "ELAIS-S1",
#                        "ECDFS", "EDFS_a", "EDFS_b"
#
USE_DDF = True  # True  → use lsstcam_fields (Deep Drilling Fields)
# False → use lsstcam_targets (SV / special targets)
key = "ECDFS"  # <-- change this to select any available field

# --- Band for the quick single-patch preview ---
BANDSEL = "u"  # preview band (used before the full mosaic loop)


# Datasetype
DATASET_COADDS = "legacy_deep_coadd"

In [ ]:
# Derived configuration (do not edit)
where_clause = f"instrument = '{instrument}'"
collectionStr = collection[-1].replace("/", "_")

# Initialise the Butler repository
butler = Butler(repo, collections=collection)
registry = butler.registry
print(f"Butler initialised | repo: {repo}")
print(f"Collection : {collection[-1]}")

In [ ]:
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)

In [ ]:
camera = butler.get("camera", collections=collection, instrument=instrument)

## Discover available deepCoadd dataset type

The name of the deepCoadd dataset type varies between pipeline versions.
This cell probes the Butler registry and selects the first available name
from a prioritised list, so the notebook works with different collections.


In [ ]:
# ----------------------------------------------------------------
# Automatic discovery of the available deepCoadd dataset type
# ----------------------------------------------------------------
# The dataset type name changed between pipeline versions:
#   Gen2  : 'deepCoadd_calexp'
#   DP1   : 'deep_coadd'
#   DP2+  : 'legacy_deep_coadd'
# We probe the registry instead of hardcoding the name so the
# notebook works across different Butler collections.

COADD_CANDIDATES = [
    "legacy_deep_coadd",
    "deep_coadd",
    "deepCoadd_calexp",
    "deepCoadd",
    "goodSeeingCoadd",
    "dcrCoadd",
    "deep_coadd_cell_predetection",
    "template_coadd",
]

# List all coadd-related dataset types actually present in the registry
all_coadd_types = [d.name for d in butler.registry.queryDatasetTypes() if "coadd" in d.name.lower()]
print("All coadd dataset types in registry:", all_coadd_types)

# Pick the first candidate that exists in the registry
COADD_DATASET = None
for name in COADD_CANDIDATES:
    if dataset_type_exists(butler, name):
        COADD_DATASET = name
        print(f"\n✔ Selected deepCoadd dataset type: '{COADD_DATASET}'")
        break

if COADD_DATASET is None:
    raise RuntimeError(
        "No recognised deepCoadd dataset type found in this Butler collection. "
        f"Available coadd types: {all_coadd_types}"
    )

In [ ]:
# print(camera.getName(),camera.getNameMap())

## List of Sky field of interest

In [ ]:
lsstcam_targets = {}
lsstcam_targets["47 Tuc"] = {"field_name": "47 Tuc Globular Cluster", "ra": 6.02, "dec": -72.08}
lsstcam_targets["Rubin SV 38 7"] = {"field_name": "Low Ecliptic Latitude Field", "ra": 37.86, "dec": 6.98}
lsstcam_targets["Fornax dSph"] = {"field_name": "Fornax Dwarf Spheroidal Galaxy", "ra": 40.0, "dec": -34.45}
lsstcam_targets["ECDFS"] = {"field_name": "Extended Chandra Deep Field South", "ra": 53.13, "dec": -28.10}
lsstcam_targets["EDFS"] = {"field_name": "Euclid Deep Field South", "ra": 59.10, "dec": -48.73}
lsstcam_targets["Rubin SV 95 -25"] = {"field_name": "Low Galactic Latitude Field", "ra": 95.00, "dec": -25.0}
lsstcam_targets["Seagull"] = {"field_name": "Seagull Nebula", "ra": 106.23, "dec": -10.51}

In [ ]:
lsstcam_fields = {}

# =========================
# Deep Drilling Fields (LSSTCam)
# =========================
lsstcam_fields["COSMOS"] = {"field_name": "COSMOS Deep Drilling Field", "ra": 150.11, "dec": 2.23}
lsstcam_fields["XMM-LSS"] = {"field_name": "XMM-LSS Deep Drilling Field", "ra": 35.57, "dec": -4.82}
lsstcam_fields["ELAIS-S1"] = {"field_name": "ELAIS-S1 Deep Drilling Field", "ra": 9.45, "dec": -44.02}
lsstcam_fields["ECDFS"] = {"field_name": "Extended Chandra Deep Field South", "ra": 52.98, "dec": -28.12}
# Euclid Deep Field South = 2 pointings LSSTCam
lsstcam_fields["EDFS_a"] = {"field_name": "Euclid Deep Field South (pointing a)", "ra": 58.90, "dec": -49.32}
lsstcam_fields["EDFS_b"] = {"field_name": "Euclid Deep Field South (pointing b)", "ra": 63.60, "dec": -47.60}

## Define the target 

In [ ]:
index_target = 0

target_info = df_targets.iloc[index_target]
display(target_info)

In [ ]:
name = target_info["simbad_id"]
field = target_info["field"]
ra = target_info["ra_deg"]
dec = target_info["dec_deg"]

## Extract the Coadds and the WCS

In [ ]:
Ntargets = len(df_targets)
ncols = 4
nrows = Ntargets // ncols
if Ntargets - nrows * ncols > 0:
    nrows += 1

print(f"{Ntargets} nrows = {nrows} , ncols = {ncols}")

In [ ]:
handles = []
# loop on targets
for index, row in df_targets.iterrows():
    name = row["simbad_id"]
    field = row["field"]
    ra = row["ra_deg"]
    dec = row["dec_deg"]

    point = geom.SpherePoint(ra, dec, geom.degrees)

    tract_info = skymap.findTract(point)
    patch_info = tract_info.findPatch(point)

    dataId = {
        "tract": tract_info.getId(),
        "patch": patch_info.getSequentialIndex(),
        "band": BANDSEL,
        "skymap": skymapName,
    }

    try:
        coadd = butler.get(COADD_DATASET, dataId=dataId)
        wcs = coadd.getWcs()
        # wcs_header = wcs.getFitsMetadata().toDict()
        # wcs_astropy = WCS(wcs_header)

        handles.append((row, coadd, wcs))
        print(f" \t ***  SUCCESS  for {name} IN {field} ==> dataId = {dataId}")
    except Exception as inst:
        print(
            f" \t >>> EXCEPTION for {name} IN {field} ==>  dataId = {dataId} ::: Exection Type", type(inst)
        )  # the exception type
        print("\t >>> \t", inst)
        print("\n")

## Plot the targets

In [ ]:
zscale = ZScaleInterval()
subset_size = 30

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(16, 4 * nrows))
axes = axes.flatten()


# Loop on targets
for idx, (target_r, coadd, wcs) in enumerate(handles):
    radec = geom.SpherePoint(target_r["ra_deg"], target_r["dec_deg"], geom.degrees)
    xy = Point2D(wcs.skyToPixel(radec))

    selection_bbox = Box2I.makeCenteredBox(xy, Extent2I(subset_size, subset_size))
    params = {"bbox": selection_bbox}

    subset = coadd[selection_bbox]

    print(f"Processed ID {target_r['simbad_id']}")

    img_array = subset.image.array
    vmin, vmax = zscale.get_limits(img_array)
    ax = axes[idx]

    ax.imshow(img_array, origin="lower")
    ax.set_title(f"{target_r['simbad_id']} ({target_r['field']})")

for i in range(len(handles), len(axes)):
    fig.delaxes(axes[i])


plt.tight_layout()
savefig(f"Rubin_cutouts_band_{BANDSEL}")
plt.show()

In [ ]:
# display.clearViewer()

In [ ]:
# setImageColormap) are “gray” and “grey”